<a href="https://colab.research.google.com/github/YYYingZZZ/SAD_2023/blob/main/CourseTrain_LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [5]:
file_path = "/content/car.txt"
columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]

df = pd.read_csv(file_path, header=None, names=columns)
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [6]:
print("資料筆數與欄位數：", df.shape)
print("\n各類別分布：")
print(df["class"].value_counts())

資料筆數與欄位數： (1728, 7)

各類別分布：
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


In [7]:
df_encoded = df.copy()

buying_map = {"low": 1, "med": 2, "high": 3, "vhigh": 4}
maint_map = {"low": 1, "med": 2, "high": 3, "vhigh": 4}
doors_map = {"2": 1, "3": 2, "4": 3, "5more": 4}
persons_map = {"2": 1, "4": 2, "more": 3}
lug_boot_map = {"small": 1, "med": 2, "big": 3}
safety_map = {"low": 1, "med": 2, "high": 3}

df_encoded["buying"] = df_encoded["buying"].map(buying_map)
df_encoded["maint"] = df_encoded["maint"].map(maint_map)
df_encoded["doors"] = df_encoded["doors"].map(doors_map)
df_encoded["persons"] = df_encoded["persons"].map(persons_map)
df_encoded["lug_boot"] = df_encoded["lug_boot"].map(lug_boot_map)
df_encoded["safety"] = df_encoded["safety"].map(safety_map)

df_encoded.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,4,4,1,1,1,1,unacc
1,4,4,1,1,1,2,unacc
2,4,4,1,1,1,3,unacc
3,4,4,1,1,2,1,unacc
4,4,4,1,1,2,2,unacc


In [8]:
X = df_encoded.drop("class", axis=1)
y = df_encoded["class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1728, 6)
y shape: (1728,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("訓練集大小：", X_train.shape)
print("測試集大小：", X_test.shape)

訓練集大小： (1382, 6)
測試集大小： (346, 6)


In [10]:
label_order = ["unacc", "acc", "good", "vgood"]

def evaluate_model(model, X_test, y_test, title="Model Result"):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=label_order)
    report = classification_report(
        y_test, y_pred,
        labels=label_order,
        zero_division=0
    )

    cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)

    print(f"\n{'='*50}")
    print(title)
    print(f"{'='*50}")
    print(f"Accuracy: {acc:.4f}\n")

    print("Confusion Matrix:")
    display(cm_df)

    print("Classification Report:")
    print(report)

    return acc, cm_df, report

In [11]:
model_v1 = LogisticRegression(random_state=42)
model_v1.fit(X_train, y_train)

acc1, cm1, report1 = evaluate_model(
    model_v1,
    X_test,
    y_test,
    title="Version 1 - Original Logistic Regression"
)


Version 1 - Original Logistic Regression
Accuracy: 0.8353

Confusion Matrix:


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,unacc,acc,good,vgood
unacc,235,6,1,0
acc,31,41,2,3
good,2,5,7,0
vgood,0,5,2,6


Classification Report:
              precision    recall  f1-score   support

       unacc       0.88      0.97      0.92       242
         acc       0.72      0.53      0.61        77
        good       0.58      0.50      0.54        14
       vgood       0.67      0.46      0.55        13

    accuracy                           0.84       346
   macro avg       0.71      0.62      0.65       346
weighted avg       0.82      0.84      0.82       346



In [12]:
model_v2 = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model_v2.fit(X_train, y_train)

acc2, cm2, report2 = evaluate_model(
    model_v2,
    X_test,
    y_test,
    title="Version 2 - Tuned Logistic Regression"
)


Version 2 - Tuned Logistic Regression
Accuracy: 0.7977

Confusion Matrix:


,unacc,acc,good,vgood
unacc,197,28,9,8
acc,11,54,5,7
good,0,0,14,0
vgood,0,1,1,11


Classification Report:
              precision    recall  f1-score   support

       unacc       0.95      0.81      0.88       242
         acc       0.65      0.70      0.68        77
        good       0.48      1.00      0.65        14
       vgood       0.42      0.85      0.56        13

    accuracy                           0.80       346
   macro avg       0.63      0.84      0.69       346
weighted avg       0.84      0.80      0.81       346



In [13]:
compare_df = pd.DataFrame({
    "Version": ["Original", "Tuned"],
    "Accuracy": [acc1, acc2]
})

display(compare_df)

,Version,Accuracy
0,Original,0.835260
1,Tuned,0.797688


In [14]:
os.makedirs("/content/result", exist_ok=True)

In [15]:
def save_markdown_report(filename, title, accuracy, cm_df, report_text):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")
        f.write("## Accuracy\n\n")
        f.write(f"{accuracy:.4f}\n\n")

        f.write("## Confusion Matrix\n\n")
        f.write("```text\n")
        f.write(cm_df.to_string())
        f.write("\n```\n\n")

        f.write("## Classification Report\n\n")
        f.write("```text\n")
        f.write(report_text)
        f.write("\n```\n")

save_markdown_report(
    "/content/result/Logistic_regression_task_1.md",
    "Logistic Regression Task 1",
    acc1,
    cm1,
    report1
)

save_markdown_report(
    "/content/result/Logistic_regression_task_2.md",
    "Logistic Regression Task 2",
    acc2,
    cm2,
    report2
)

print("Markdown 報告已輸出到 /content/result/")

Markdown 報告已輸出到 /content/result/
